[LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)  
[LangGraph Agentic RAG Tutorial](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_agentic_rag/#7-assemble-the-graph)

# 1. Preparar documentos

In [1]:
import json
from pathlib import Path
from datetime import datetime
from langchain.docstore.document import Document
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 📥 Ruta del archivo enriquecido de votaciones
loader = JSONLoader(
    file_path=Path("../data/voting_docs_enriched.json"),
    jq_schema=".[]",
    text_content=False  # Cada `Document.page_content` será la cadena JSON
)

docs_raw = loader.load()

def normalize(doc: Document) -> tuple[Document, str]:
    # 🔑 Convertir el texto JSON a dict
    rec = json.loads(doc.page_content) if isinstance(doc.page_content, str) else doc.page_content

    texto = (
        f"Votación del {rec.get('fecha_larga')}.\n"
        f"Legislatura: {rec.get('legislatura')} – Congreso {rec.get('periodo_congreso')}.\n"
        f"Periodo anual: {rec.get('periodo_anual')}.\n"
        f"Asunto: {rec.get('asunto')}.\n"
        f"Presidente de la sesión: {rec.get('presidente')}.\n"
        f"URL: {rec.get('url')}"
    )

    # Timestamp y desgloses para filtros
    dt = datetime.fromisoformat(rec["fecha"])
    fecha_ts = int(dt.timestamp() * 1000)  # milisegundos Unix

    # 🗂️ Crear metadata limpio asegurando la URL y el ID original
    metadata = {
        # Identificación
        "doc_id": rec["id"],
        "sesion": rec["sesion"],
        # Temporal
        "fecha_iso": rec["fecha"],
        "fecha_ts":  fecha_ts,
        "anio":      dt.year,
        "mes":       dt.month,
        "dia":       dt.day,
        "hora":      rec["hora"],
        "fecha_larga": rec["fecha_larga"],
        "fecha_corta": rec["fecha_corta"],
        # Legislatura / periodos
        "legislatura_txt": rec["legislatura"],
        "n_legislatura":   rec["n_legislatura"],
        "periodo_congreso_inicio": rec["periodo_congreso_inicio"],
        "periodo_congreso_fin":    rec["periodo_congreso_fin"],
        "periodo_anual_inicio":    rec["periodo_anual_inicio"],
        "periodo_anual_fin":       rec["periodo_anual_fin"],
        # Navegación
        "page": rec["page"],
        "url":  rec["url"],
        # Tipo de documento y versión
        "doctype": "voting_header",
        "version": datetime.now().strftime("%Y.%m.%d.%H.%M"),
    }

    # Retornar el documento Y su ID personalizado
    return Document(page_content=texto, metadata=metadata), rec.get("id")

# 🔧 Procesar documentos y extraer IDs personalizados
docs_and_ids = [normalize(d) for d in docs_raw]
docs = [doc for doc, _ in docs_and_ids]
custom_ids = [doc_id for _, doc_id in docs_and_ids]

# 🔍 Dividir documentos si exceden cierto tamaño
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

print(f"{len(docs_raw)=}  {len(split_docs)=}")
print(f"Primeros 3 IDs personalizados: {custom_ids[:3]}")
print(f"Documentos cargados: {len(docs_raw)}")
print(f"Documentos divididos: {len(split_docs)}")
print(f"Ejemplo de metadata:")
print(json.dumps(split_docs[0].metadata, indent=2))

len(docs_raw)=7591  len(split_docs)=7591
Primeros 3 IDs personalizados: ['f9e959cb-2241-4c73-b101-381740a24dce', '695dc12a-1540-495b-839b-c442f568bfbc', 'a117d4f5-c7dd-4156-94b6-448fe6c7ddf8']
Documentos cargados: 7591
Documentos divididos: 7591
Ejemplo de metadata:
{
  "doc_id": "f9e959cb-2241-4c73-b101-381740a24dce",
  "sesion": "937_pp2006_2011_pa2009_2010_leg1_page_11",
  "fecha_iso": "2009-10-07T15:25:00",
  "fecha_ts": 1254947100000,
  "anio": 2009,
  "mes": 10,
  "dia": 7,
  "hora": "15:25",
  "fecha_larga": "07 de octubre del 2009",
  "fecha_corta": "07 oct 2009",
  "legislatura_txt": "Primera Legislatura Ordinaria 2009-2010",
  "n_legislatura": 1,
  "periodo_congreso_inicio": 2006,
  "periodo_congreso_fin": 2011,
  "periodo_anual_inicio": 2009,
  "periodo_anual_fin": 2010,
  "page": 11,
  "url": "http://localhost:8080/votacion/937_pp2006_2011_pa2009_2010_leg1_page_11.png",
  "doctype": "voting_header",
  "version": "2025.09.02.04.07"
}


In [2]:
docs[0]

Document(metadata={'doc_id': 'f9e959cb-2241-4c73-b101-381740a24dce', 'sesion': '937_pp2006_2011_pa2009_2010_leg1_page_11', 'fecha_iso': '2009-10-07T15:25:00', 'fecha_ts': 1254947100000, 'anio': 2009, 'mes': 10, 'dia': 7, 'hora': '15:25', 'fecha_larga': '07 de octubre del 2009', 'fecha_corta': '07 oct 2009', 'legislatura_txt': 'Primera Legislatura Ordinaria 2009-2010', 'n_legislatura': 1, 'periodo_congreso_inicio': 2006, 'periodo_congreso_fin': 2011, 'periodo_anual_inicio': 2009, 'periodo_anual_fin': 2010, 'page': 11, 'url': 'http://localhost:8080/votacion/937_pp2006_2011_pa2009_2010_leg1_page_11.png', 'doctype': 'voting_header', 'version': '2025.09.02.04.07'}, page_content='Votación del 07 de octubre del 2009.\nLegislatura: Primera Legislatura Ordinaria 2009-2010 – Congreso 2006-2011.\nPeriodo anual: 2009-2010.\nAsunto: MISION MOCION 8445 ,CONFORMACION DE UNA COMISION ESPECIAL ENCARGADA DEL CONTROL, SEGUIMIENTO Y ALUACION DEL PLAN NACIONAL DE LUCHA CONTRA LA CORRUPCION.\nPresidente de 

# 2. Indexar documentos

https://python.langchain.com/api_reference/qdrant/qdrant/langchain_qdrant.qdrant.QdrantVectorStore.html

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# %%
# --- BLOQUE ÚNICO corregido ---------------------------------
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# 1️⃣ Embeddings de OpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2️⃣ Cliente al contenedor Docker
qdrant_client = QdrantClient(host="localhost", port=6333)

# 3️⃣ Crear (o recrear) la colección con la dimensión correcta
vector_dim = len(embeddings.embed_query("ping"))  # 1536 p/text-embedding-3-small
collection_name = "voting_docs"

#! ⚠️ Si la colección ya existe, se eliminará y se creará de nuevo
#! Para producción agregar el **if not**
if qdrant_client.collection_exists(collection_name):
    qdrant_client.delete_collection(collection_name=collection_name)

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
)

# 🆕 3.1  Índices de payload (numeric / keyword) --------------
numeric_fields = [
    "fecha_ts",
    "anio",
    "mes",
    "dia",
    "n_legislatura",
    "periodo_congreso_inicio",
    "periodo_congreso_fin",
    "periodo_anual_inicio",
    "periodo_anual_fin",
    "page",
]
for fld in numeric_fields:
    qdrant_client.create_payload_index(
        collection_name=collection_name, field_name=fld, field_schema="integer"
    )

# Si vas a mezclar tipos de documentos en la colección
qdrant_client.create_payload_index(
    collection_name=collection_name, field_name="doctype", field_schema="keyword"
)
# -------------------------------------------------------------

# 4️⃣ Insertar puntos con payload PLANO ----------------------
from qdrant_client.http.models import PointStruct

BATCH = 64
for i in range(0, len(split_docs), BATCH):
    chunk_docs = split_docs[i : i + BATCH]
    chunk_ids = custom_ids[i : i + BATCH]

    # a) Embeddings
    vectors = embeddings.embed_documents([d.page_content for d in chunk_docs])

    # b) Construir puntos con payload plano (sin clave "metadata")
    points = [
        PointStruct(
            id=uid,
            vector=vec,
            payload={"page_content": doc.page_content, **doc.metadata},
        )
        for uid, vec, doc in zip(chunk_ids, vectors, chunk_docs)
    ]

    qdrant_client.upsert(collection_name=collection_name, points=points)

print(f"✅ Documentos indexados: {len(split_docs)} puntos en '{collection_name}'")

# 5️⃣ Instanciar el VectorStore sobre la colección existente
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=collection_name,
    embedding=embeddings,
)

print("🔗 VectorStore listo para consultas (asimilarity_search, etc.)")

✅ Documentos indexados: 7591 puntos en 'voting_docs'
🔗 VectorStore listo para consultas (asimilarity_search, etc.)


In [4]:
# 🔍 Verificar que ahora usamos los IDs del JSON original
print("IDs originales vs IDs asignados:")
for i in range(3):
    print(f"  Original: {custom_ids[i]}")
    print(f"  Asignado: {split_docs[i].metadata['doc_id']}")
    print(f"  ¿Coinciden? {custom_ids[i] == split_docs[i].metadata['doc_id']}")
    print("---")

IDs originales vs IDs asignados:
  Original: f9e959cb-2241-4c73-b101-381740a24dce
  Asignado: f9e959cb-2241-4c73-b101-381740a24dce
  ¿Coinciden? True
---
  Original: 695dc12a-1540-495b-839b-c442f568bfbc
  Asignado: 695dc12a-1540-495b-839b-c442f568bfbc
  ¿Coinciden? True
---
  Original: a117d4f5-c7dd-4156-94b6-448fe6c7ddf8
  Asignado: a117d4f5-c7dd-4156-94b6-448fe6c7ddf8
  ¿Coinciden? True
---


In [5]:
# 📋 Verificar estructura del documento con nuevos campos
print("📄 Estructura del primer documento:")
print(f"🆔 doc_id: {docs[0].metadata['doc_id']}")
print(f"🔢 version: {docs[0].metadata['version']}")
print(f"🗓️ fecha_larga: {docs[0].metadata['fecha_larga']}")
print(f"📊 metadata completos: {list(docs[0].metadata.keys())}")
print()
print("📄 Documento completo:")
print(docs[0])


📄 Estructura del primer documento:
🆔 doc_id: f9e959cb-2241-4c73-b101-381740a24dce
🔢 version: 2025.09.02.04.07
🗓️ fecha_larga: 07 de octubre del 2009
📊 metadata completos: ['doc_id', 'sesion', 'fecha_iso', 'fecha_ts', 'anio', 'mes', 'dia', 'hora', 'fecha_larga', 'fecha_corta', 'legislatura_txt', 'n_legislatura', 'periodo_congreso_inicio', 'periodo_congreso_fin', 'periodo_anual_inicio', 'periodo_anual_fin', 'page', 'url', 'doctype', 'version']

📄 Documento completo:
page_content='Votación del 07 de octubre del 2009.
Legislatura: Primera Legislatura Ordinaria 2009-2010 – Congreso 2006-2011.
Periodo anual: 2009-2010.
Asunto: MISION MOCION 8445 ,CONFORMACION DE UNA COMISION ESPECIAL ENCARGADA DEL CONTROL, SEGUIMIENTO Y ALUACION DEL PLAN NACIONAL DE LUCHA CONTRA LA CORRUPCION.
Presidente de la sesión: Alva Castro, Luis Juan.
URL: http://localhost:8080/votacion/937_pp2006_2011_pa2009_2010_leg1_page_11.png' metadata={'doc_id': 'f9e959cb-2241-4c73-b101-381740a24dce', 'sesion': '937_pp2006_2011_